# The Seven Layers of AI: Layer 2: Machine Learning

This notebook contains every example program from **Chapter 3** of the study guide *The Seven Layers of AI*, in the order they appear. Each program comes with the explanation that introduces it in the guide, the output printed in the guide, what to notice in that output, and the Investigate tasks.

**How to use it:** run the cells from top to bottom. Each program is self-contained, so you can also run any single program on its own after the setup cell. Change the code freely: the Investigate tasks suggest where to start.

## Programs in this notebook

1. `l2_workflow.py`: the machine learning workflow, and two ways to fool yourself
2. `l2_linear_regression_worked.py`: Code example: linear regression, calculated four ways
3. `l2_logistic.py`: classifying tumours
4. `l2_trees.py`: Gradient boosting
5. `l2_knn_svm.py`: k-NN from scratch and SVM kernels
6. `l2_naive_bayes.py`: a spam filter
7. `l2_unsupervised.py`: k-means from scratch and PCA
8. `l2_overfitting.py`: overfitting and cross-validation
9. `l2_evaluation.py`: the accuracy trap
10. `l2_pipeline.py`: an end-to-end churn model
11. `l2_qlearning.py`: Q-learning in a grid world
12. `l2_fairness.py`: bias through a proxy

## Setup

The programs in this notebook use: `numpy pandas scikit-learn matplotlib joblib`. If any are missing, uncomment the line below and run it once.

In [1]:
# %pip install numpy pandas scikit-learn matplotlib joblib
%matplotlib inline

## 1. The machine learning workflow, and two ways to fool yourself

**File:** `l2_workflow.py`

In [2]:
"""
Layer 2 example: the machine learning workflow, and two ways to fool yourself
File: l2_workflow.py
 
Purpose:
  Walks through the standard ML workflow with a scikit-learn Pipeline, then
  shows the two commonest ways to fool yourself: leaking information from the
  test data into preprocessing, and choosing a model by its test score.
 
How it works:
  Part 1  A complete workflow (stages 1 to 6 of Figure 10) using a Pipeline
  Part 2  Leakage: preprocessing fitted before the split
  Part 3  Tuning on the test set
 
What to look for in the output:
  - Cross-validation picks C = 10; the test set is then used once (0.981).
  - On pure noise, feature selection before splitting reports 0.88, while the
    honest Pipeline gives about 0.50.
  - The model 'tuned' on the test set scores 0.62 there but only 0.50 on new
    data: the test score was luck, not skill.
 
Requires: pip install scikit-learn numpy
"""
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
 
rng = np.random.default_rng(42)
 
# =============================================================================
# PART 1: the workflow, stage by stage
# =============================================================================
print("PART 1: a complete workflow")
 
# ---- Stage 1: collect data ---------------------------------------------------
data = load_wine()
X, y = data.data.copy(), data.target
 
# ---- Stage 2: clean and explore ----------------------------------------------
# Real data has gaps, so knock out 5% of values at random to simulate them.
mask = rng.random(X.shape) < 0.05
X[mask] = np.nan
print(f"  missing values: {np.isnan(X).sum()} of {X.size}")
print(f"  class counts:   {np.bincount(y)}  (roughly balanced)")
 
# ---- Stage 4: split, and lock the test set away -------------------------------
# Note: we split BEFORE fitting any preprocessing (stage 3 happens in the pipeline).
X_train, X_test, y_train, y_test = train_test_split(
    # stratify keeps class proportions the same in both sets
    X, y, test_size=0.3, random_state=1, stratify=y)
 
# ---- Stage 3: engineer features, inside a Pipeline ----------------------------
# A Pipeline chains the steps. When it is fitted, each step learns ONLY from the
# data it is given: the median, mean and standard deviation come from training rows.
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),     # fill gaps with the column median
    ("scale",  StandardScaler()),                     # mean 0, standard deviation 1
    ("model",  LogisticRegression(max_iter=1000)),
])
 
# ---- Stage 5: train and tune, judged by cross-validation on TRAINING data ------
# C controls regularisation strength: a hyperparameter we choose, not one it learns.
search = GridSearchCV(pipe, {"model__C": [0.01, 0.1, 1, 10, 100]}, cv=5)
search.fit(X_train, y_train)
for C, s in zip(search.cv_results_["param_model__C"],
                search.cv_results_["mean_test_score"]):
    print(f"  C = {C:<6}  cross-validation accuracy {s:.3f}")
print(f"  chosen: C = {search.best_params_['model__C']}")
 
# ---- Stage 6: final test, ONCE --------------------------------------------------
print(f"  final test accuracy (used once): {search.score(X_test, y_test):.3f}")
 
# =============================================================================
# PART 2: leakage through preprocessing
# =============================================================================
# The data is pure noise: 100 people, 5,000 random measurements, random labels.
# There is NOTHING to learn, so an honest method must score about 50%.
print("\nPART 2: leakage (the data is pure noise, so honest accuracy is about 0.50)")
Xn = rng.normal(size=(100, 5000))
yn = rng.integers(0, 2, 100)
 
# WRONG: choose the 20 'best' features using ALL the data, then cross-validate.
# The selector has already seen the labels of the rows used for testing.
X_sel = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, yn, cv=5).mean()
print(f"  selection before splitting:  {wrong:.2f}   <- looks brilliant, is fake")
 
# RIGHT: put selection inside a Pipeline, so each fold selects from its own
# training rows.
right_pipe = Pipeline([("select", SelectKBest(f_classif, k=20)),
                       ("model",  LogisticRegression(max_iter=1000))])
right = cross_val_score(right_pipe, Xn, yn, cv=5).mean()
print(f"  selection inside a Pipeline: {right:.2f}   <- the honest answer")
 
# =============================================================================
# PART 3: tuning on the test set
# =============================================================================
# Again pure noise. Try 200 random tree settings and keep whichever scores best
# on a small test set of 40 rows. Then check that 'winner' on brand-new data
# from the same source.
print("\nPART 3: tuning on the test set (pure noise again)")
Xa = rng.normal(size=(200, 20)); ya = rng.integers(0, 2, 200)
Xtr, Xte, ytr, yte = train_test_split(Xa, ya, test_size=40, random_state=0)
Xnew = rng.normal(size=(5000, 20)); ynew = rng.integers(0, 2, 5000)   # the 'real world'
 
best_score, best_model = 0, None
for seed in range(200):
    m = DecisionTreeClassifier(max_depth=int(rng.integers(1, 8)),
                               max_features=int(rng.integers(1, 21)),
                               random_state=seed).fit(Xtr, ytr)
    s = m.score(Xte, yte)                     # WRONG: using the test set to choose
    if s > best_score:
        best_score, best_model = s, m
print(f"  best of 200 on the test set:  {best_score:.2f}   <- reported result")
print(f"  same model on new data:       "
      f"{best_model.score(Xnew, ynew):.2f}   <- what users get")

PART 1: a complete workflow
  missing values: 111 of 2314
  class counts:   [59 71 48]  (roughly balanced)


  C = 0.01    cross-validation accuracy 0.976
  C = 0.1     cross-validation accuracy 0.976
  C = 1.0     cross-validation accuracy 0.984
  C = 10.0    cross-validation accuracy 0.992
  C = 100.0   cross-validation accuracy 0.992
  chosen: C = 10
  final test accuracy (used once): 0.981

PART 2: leakage (the data is pure noise, so honest accuracy is about 0.50)


  selection before splitting:  0.88   <- looks brilliant, is fake


  selection inside a Pipeline: 0.52   <- the honest answer

PART 3: tuning on the test set (pure noise again)


  best of 200 on the test set:  0.62   <- reported result
  same model on new data:       0.50   <- what users get


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
PART 1: a complete workflow
  missing values: 111 of 2314
  class counts:   [59 71 48]  (roughly balanced)
  C = 0.01    cross-validation accuracy 0.976
  C = 0.1     cross-validation accuracy 0.976
  C = 1.0     cross-validation accuracy 0.984
  C = 10.0    cross-validation accuracy 0.992
  C = 100.0   cross-validation accuracy 0.992
  chosen: C = 10
  final test accuracy (used once): 0.981
PART 2: leakage (the data is pure noise, so honest accuracy is about 0.50)
  selection before splitting:  0.88   <- looks brilliant, is fake
  selection inside a Pipeline: 0.52   <- the honest answer
PART 3: tuning on the test set (pure noise again)
  best of 200 on the test set:  0.62   <- reported result
  same model on new data:       0.50   <- what users get
```

## 2. Linear regression, calculated four ways

**File:** `l2_linear_regression_worked.py`

The program l2_linear_regression_worked.py reproduces every number in this chapter, calculating the same line four ways: by hand from the means, with the normal equation, with gradient descent and with scikit-learn. It then shows the effect of a single outlier and compares ridge and lasso, including lasso switching off features on a real medical dataset.

In [3]:
"""
Layer 2 example: linear regression, calculated four ways
File: l2_linear_regression_worked.py
 
Purpose:
  Linear regression is the simplest learning algorithm: fit a straight line
  that minimises squared error. Calculating it four ways (by hand, by the
  normal equation, by gradient descent, by scikit-learn) shows they agree,
  and reveals the ideas later models build on: loss, gradients, scaling and
  regularisation.
 
How it works:
  Reproduces every number in the Worked calculations section:
  Part 1  the best line by hand, from the means            (Calculation 1)
  Part 2  predictions, residuals, MSE, RMSE and R-squared  (Calculations 2 to 4)
  Part 3  the normal equation with NumPy                   (Calculation 5)
  Part 4  gradient descent, step by step                   (Calculation 6)
  Part 5  scikit-learn, and the effect of one outlier
  Part 6  ridge and lasso                                   (Calculation 7)
 
What to look for in the output:
  - Every method finds the same line: price = 2.110 x area + 44.79.
  - Gradient descent on raw area is unstable (MSE jumps to 14,602 after one
    step), but on standardised area rate 0.1 converges, 0.01 is too slow and
    1.05 diverges.
  - One bad record drags the weight from 2.110 to 0.765.
  - Larger lasso alphas set more diabetes weights to exactly zero (0 -> 5).
 
Requires: pip install numpy scikit-learn
"""
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.datasets import load_diabetes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
 
# The five houses used throughout the section
# x: floor area in square metres
area  = np.array([50, 70, 80, 100, 120], dtype=float)
# y: price in thousands of pounds
price = np.array([150, 190, 220, 250, 300], dtype=float)
n = len(area)
 
# =============================================================================
# PART 1: the best line by hand (Calculation 1)
# =============================================================================
print("PART 1: the best line from the means")
x_bar, y_bar = area.mean(), price.mean()                  # 84 and 222
# distance of each value from its mean
dx, dy = area - x_bar, price - y_bar
# 6160: do x and y rise together?
sxy = np.sum(dx * dy)
sxx = np.sum(dx ** 2)                                     # 2920: how spread out is x?
w = sxy / sxx                                             # weight (slope)
# bias: the line passes through (x_bar, y_bar)
b = y_bar - w * x_bar
print(f"  means: x = {x_bar:.0f}, y = {y_bar:.0f}")
print(f"  sum (x-x_bar)(y-y_bar) = {sxy:.0f},  sum (x-x_bar)^2 = {sxx:.0f}")
print(f"  w = {sxy:.0f} / {sxx:.0f} = {w:.3f}")
print(f"  b = {y_bar:.0f} - {w:.3f} x {x_bar:.0f} = {b:.2f}")
print(f"  line: price = {w:.2f} x area + {b:.1f}")
 
# =============================================================================
# PART 2: predictions, residuals and how good the fit is (Calculations 2 to 4)
# =============================================================================
print("\nPART 2: residuals, MSE and R-squared")
pred = w * area + b                                       # prediction for every house
# residual = true value - prediction
resid = price - pred
print("  area  price   predicted  residual  residual^2")
for x, y, p, r in zip(area, price, pred, resid):
    print(f"  {x:4.0f}  {y:5.0f}   {p:9.2f}  {r:+8.2f}  {r**2:10.2f}")
# sum of squared residuals: 84.93
sse = np.sum(resid ** 2)
mse = sse / n                                             # mean squared error
# back in the original units (GBP k)
rmse = np.sqrt(mse)
# error of always predicting the mean: 13,080
sst = np.sum(dy ** 2)
# fraction of that error the line removes
r2 = 1 - sse / sst
print(f"  residuals sum to {resid.sum():.10f}  "
      f"(always zero for least squares with a bias)")
print(f"  MSE  = {sse:.2f} / {n} = {mse:.2f}")
print(f"  RMSE = sqrt({mse:.2f}) = {rmse:.2f}  "
      f"(typical error about GBP {rmse*1000:,.0f})")
print(f"  R^2  = 1 - {sse:.2f} / {sst:.0f} = {r2:.4f}")
print(f"  prediction for 90 m^2: {w * 90 + b:.2f}  (about GBP {w*90+b:.0f}k)")
 
# =============================================================================
# PART 3: the normal equation (Calculation 5)
# =============================================================================
print("\nPART 3: the normal equation, w = (X^T X)^-1 X^T y")
# each row is [1, x]; the 1 multiplies the bias
X = np.column_stack([np.ones(n), area])
XtX = X.T @ X                                             # [[5, 420], [420, 38200]]
Xty = X.T @ price                                         # [1110, 99400]
print(f"  X^T X = {XtX.tolist()}")
print(f"  X^T y = {Xty.tolist()}")
print(f"  determinant = {np.linalg.det(XtX):,.0f}")
b_ne, w_ne = np.linalg.inv(XtX) @ Xty                     # same answer as Part 1
print(f"  bias = {b_ne:.2f}, weight = {w_ne:.3f}")
# In practice use np.linalg.solve or lstsq: inverting a matrix directly is slower
# and less accurate when features are strongly correlated.
 
# =============================================================================
# PART 4: gradient descent (Calculation 6)
# =============================================================================
print("\nPART 4: gradient descent")
 
def gradients(w, b, x, y):
    """Gradients of the MSE with respect to w and b."""
    # residuals for the current line
    err = y - (w * x + b)
    grad_w = -2 / len(x) * np.sum(x * err)                # how MSE changes as w changes
    grad_b = -2 / len(x) * np.sum(err)                    # how MSE changes as b changes
    return grad_w, grad_b
 
def mse_of(w, b, x, y):
    return np.mean((y - (w * x + b)) ** 2)
 
# One step on the RAW data, starting from w = 0, b = 0
gw, gb = gradients(0, 0, area, price)
print(f"  start: w = 0, b = 0, MSE = {mse_of(0, 0, area, price):,.0f}")
print(f"  gradients: w {gw:,.0f},  b {gb:,.0f}   (w's gradient is ~90x larger)")
for lr in (0.00005, 0.0001):
    # new = old - learning rate x gradient
    w1, b1 = 0 - lr * gw, 0 - lr * gb
    m1 = mse_of(w1, b1, area, price)
    print(f"  one step, learning rate {lr}: "
          f"w = {w1:.3f}, b = {b1:.4f}, MSE = {m1:,.0f}")
 
# The fix: standardise the feature so one learning rate suits both w and b
z = (area - x_bar) / area.std()                           # mean 0, standard deviation 1
print("\n  on standardised area, 40 steps from zero:")
for lr in (0.1, 0.01, 1.05):
    wz, bz = 0.0, 0.0
    for step in range(40):
        gw, gb = gradients(wz, bz, z, price)
        wz, bz = wz - lr * gw, bz - lr * gb
    final = mse_of(wz, bz, z, price)
    # Convert back to the original units so the line can be compared with Part 1
    w_orig = wz / area.std()
    b_orig = bz - wz * x_bar / area.std()
    verdict = ("converged" if final < 17.1
               else ("too slow" if final < 51900 else "DIVERGED"))
    print(f"    learning rate {lr:<5}: MSE {final:>14,.1f}  "
          f"line {w_orig:8.3f} x area + {b_orig:9.2f}  {verdict}")
 
# =============================================================================
# PART 5: scikit-learn, and the effect of one outlier
# =============================================================================
print("\nPART 5: scikit-learn")
# scikit-learn wants a 2-D feature array
model = LinearRegression().fit(area.reshape(-1, 1), price)
r2_sk = model.score(area.reshape(-1, 1), price)
print(f"  weight {model.coef_[0]:.3f}, bias {model.intercept_:.2f}, R^2 {r2_sk:.4f}")
 
# Add a sixth house of 60 m^2 at GBP 400k: a data-entry error
area6, price6 = np.append(area, 60), np.append(price, 400)
bad = LinearRegression().fit(area6.reshape(-1, 1), price6)
print(f"  with one bad record: weight {bad.coef_[0]:.3f}, bias {bad.intercept_:.2f}")
before, after = model.predict([[90]])[0], bad.predict([[90]])[0]
print(f"  prediction for 90 m^2 moves from {before:.0f} to {after:.0f}")
 
# =============================================================================
# PART 6: ridge and lasso (Calculation 7)
# =============================================================================
print("\nPART 6: ridge and lasso on the houses")
X1 = area.reshape(-1, 1)
for name, m in [("least squares  ", LinearRegression()),
                ("ridge  a=1000  ", Ridge(alpha=1000)),
                ("lasso  a=500   ", Lasso(alpha=500)),
                ("lasso  a=1232  ", Lasso(alpha=1232))]:
    m.fit(X1, price)
    print(f"  {name} weight {m.coef_[0]:6.3f}   bias {m.intercept_:6.1f}")
 
# With many features, lasso switches some off completely.
# Diabetes data: 442 patients, 10 measurements, target = disease progression
# after a year.
print("\n  diabetes data (10 features): how many weights are exactly zero?")
Xd, yd = load_diabetes(return_X_y=True)
for alpha in (0.01, 0.1, 1.0, 5.0):
    # scale first: the penalty must be fair
    pipe = make_pipeline(StandardScaler(), Lasso(alpha=alpha))
    pipe.fit(Xd, yd)
    coefs = pipe[-1].coef_
    print(f"    lasso alpha {alpha:<4}: {np.sum(coefs == 0):2d} of 10 weights are zero")

PART 1: the best line from the means
  means: x = 84, y = 222
  sum (x-x_bar)(y-y_bar) = 6160,  sum (x-x_bar)^2 = 2920
  w = 6160 / 2920 = 2.110
  b = 222 - 2.110 x 84 = 44.79
  line: price = 2.11 x area + 44.8

PART 2: residuals, MSE and R-squared
  area  price   predicted  residual  residual^2
    50    150      150.27     -0.27        0.08
    70    190      192.47     -2.47        6.08
    80    220      213.56     +6.44       41.45
   100    250      255.75     -5.75       33.10
   120    300      297.95     +2.05        4.22
  residuals sum to -0.0000000000  (always zero for least squares with a bias)
  MSE  = 84.93 / 5 = 16.99
  RMSE = sqrt(16.99) = 4.12  (typical error about GBP 4,121)
  R^2  = 1 - 84.93 / 13080 = 0.9935
  prediction for 90 m^2: 234.66  (about GBP 235k)

PART 3: the normal equation, w = (X^T X)^-1 X^T y
  X^T X = [[5.0, 420.0], [420.0, 38200.0]]
  X^T y = [1110.0, 99400.0]
  determinant = 14,600
  bias = 44.79, weight = 2.110

PART 4: gradient descent
  start: 

  weight 2.110, bias 44.79, R^2 0.9935
  with one bad record: weight 0.765, bias 190.49
  prediction for 90 m^2 moves from 235 to 259

PART 6: ridge and lasso on the houses
  least squares   weight  2.110   bias   44.8
  ridge  a=1000   weight  1.571   bias   90.0
  lasso  a=500    weight  1.253   bias  116.7
  lasso  a=1232   weight  0.000   bias  222.0

  diabetes data (10 features): how many weights are exactly zero?
    lasso alpha 0.01:  0 of 10 weights are zero
    lasso alpha 0.1 :  1 of 10 weights are zero
    lasso alpha 1.0 :  3 of 10 weights are zero
    lasso alpha 5.0 :  5 of 10 weights are zero


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
PART 1: the best line from the means
  means: x = 84, y = 222
  sum (x-x_bar)(y-y_bar) = 6160,  sum (x-x_bar)^2 = 2920
  w = 6160 / 2920 = 2.110
  b = 222 - 2.110 x 84 = 44.79
  line: price = 2.11 x area + 44.8
 
PART 2: residuals, MSE and R-squared
  area  price   predicted  residual  residual^2
    50    150      150.27     -0.27        0.08
    70    190      192.47     -2.47        6.08
    80    220      213.56     +6.44       41.45
   100    250      255.75     -5.75       33.10
   120    300      297.95     +2.05        4.22
  residuals sum to -0.0000000000  (always zero for least squares with a bias)
  MSE  = 84.93 / 5 = 16.99
  RMSE = sqrt(16.99) = 4.12  (typical error about GBP 4,121)
  R^2  = 1 - 84.93 / 13080 = 0.9935
  prediction for 90 m^2: 234.66  (about GBP 235k)
 
PART 3: the normal equation, w = (X^T X)^-1 X^T y
  X^T X = [[5.0, 420.0], [420.0, 38200.0]]
  X^T y = [1110.0, 99400.0]
  determinant = 14,600
  bias = 44.79, weight = 2.110
 
PART 4: gradient descent
  start: w = 0, b = 0, MSE = 51,900
  gradients: w -39,760,  b -444   (w's gradient is ~90x larger)
  one step, learning rate 5e-05: w = 1.988, b = 0.0222, MSE = 3,049
  one step, learning rate 0.0001: w = 3.976, b = 0.0444, MSE = 14,602
 
  on standardised area, 40 steps from zero:
    learning rate 0.1  : MSE           17.0  line    2.109 x area +     44.79  converged
    learning rate 0.01 : MSE       10,323.5  line    1.169 x area +     24.83  too slow
    learning rate 1.05 : MSE  106,277,193.4  line  -93.369 x area +  -1982.57  DIVERGED
 
PART 5: scikit-learn
  weight 2.110, bias 44.79, R^2 0.9935
  with one bad record: weight 0.765, bias 190.49
  prediction for 90 m^2 moves from 235 to 259
 
PART 6: ridge and lasso on the houses
  least squares   weight  2.110   bias   44.8
  ridge  a=1000   weight  1.571   bias   90.0
  lasso  a=500    weight  1.253   bias  116.7
  lasso  a=1232   weight  0.000   bias  222.0
 
  diabetes data (10 features): how many weights are exactly zero?
    lasso alpha 0.01:  0 of 10 weights are zero
    lasso alpha 0.1 :  1 of 10 weights are zero
    lasso alpha 1.0 :  3 of 10 weights are zero
    lasso alpha 5.0 :  5 of 10 weights are zero
```

**What the output shows**

Every value matches the Worked calculations. Three results deserve a closer look:

Part 4: the one-step results on raw data show why features are standardised. After standardising, a learning rate of 0.1 reaches exactly the same line as Part 1 in 40 steps, 0.01 is still far from it, and 1.05 diverges to a nonsensical line.

Part 5: one wrong record out of six cuts the weight from 2.11 to 0.77, which is the sensitivity to outliers described earlier, in numbers.

Part 6: on the diabetes data, raising lasso's alpha from 0.01 to 5.0 switches off half of the ten features. Ridge never sets a weight exactly to zero; try replacing Lasso with Ridge in that loop to confirm it.

## 3. Classifying tumours

**File:** `l2_logistic.py`

The example uses the Wisconsin diagnostic breast cancer dataset, in which 30 measurements were computed from digitised images of cell nuclei (Street et al., 1993). It is included with scikit-learn for teaching.

In [4]:
"""
Layer 2 example: logistic regression for classification.
File: l2_logistic.py
 
Purpose:
  Logistic regression is the standard starting point for classification. It
  outputs a PROBABILITY rather than just a label, and its weights can be read
  to see which features push a prediction one way or the other. Here it tells
  malignant tumours from benign ones.
 
  Dataset: Wisconsin breast cancer (built into scikit-learn), 569 tumours,
  30 numeric measurements, label 0 = malignant, 1 = benign.
 
  Logistic regression computes  z = w1*x1 + ... + wn*xn + b  and passes it
  through the sigmoid function  p = 1 / (1 + e^-z)  to get a PROBABILITY.
  A threshold (default 0.5) turns the probability into a class.
 
How it works:
  1. Load the data and split it into training and test sets (stratified).
  2. Build a pipeline: standardise the features, then fit logistic regression.
  3. Report test accuracy and the predicted probabilities for five patients.
  4. List the features with the largest negative and positive weights.
  5. Recompute one probability by hand with the sigmoid to show the maths.
 
What to look for in the output:
  - Test accuracy of about 96% from a simple linear model.
  - Probabilities close to 0 or 1: the model is confident on these patients.
  - Negative weights (e.g. radius error, worst texture) point to malignant.
  - The hand-computed sigmoid (0.996) equals predict_proba for patient 0.
 
Requires: pip install numpy scikit-learn
"""
 
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
 
# ---- Load and split the data ---------------------------------------------
# X is a 569 x 30 matrix of measurements; y holds the 0/1 labels.
data = load_breast_cancer()
X, y = data.data, data.target
 
# Hold back 25% of patients as a test set the model never trains on, so we
# can measure how well it generalises to new patients.
# stratify=y keeps the malignant/benign ratio the same in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
 
# ---- Build and train the model --------------------------------------------
# StandardScaler rescales each feature to mean 0 and standard deviation 1.
# Without it, features measured in large units (e.g. area) would dominate
# and the optimiser would converge slowly.
# A PIPELINE chains preprocessing and model. The scaler is fitted on the
# TRAINING data only, then reused on test data. Fitting it on all the data
# would leak information about the test set into training ('data leakage').
# max_iter=1000 gives the optimiser enough steps to minimise the log loss
# (the penalty for confident wrong probabilities). LogisticRegression also
# applies mild L2 regularisation by default (C=1.0) to keep weights small.
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
 
# Accuracy = fraction of test patients classified correctly (threshold 0.5).
print("Test accuracy:", round(accuracy_score(y_test, model.predict(X_test)), 3))
 
# ---- Probabilities, not just labels -------------------------------------
# predict_proba returns one probability per class; each row sums to 1.
# A probability lets a doctor see HOW sure the model is, and lets us move
# the threshold if missing a malignant case is worse than a false alarm.
probs = model.predict_proba(X_test[:5])      # column 0 = malignant, 1 = benign
for p, true in zip(probs, y_test[:5]):
    print(f"P(benign) = {p[1]:.3f}   actual: {data.target_names[true]}")
 
# ---- Interpreting the learned weights -----------------------------------
# Because features were standardised, weight sizes are comparable.
# Negative weight: larger values push towards class 0 (malignant).
# named_steps lets us reach inside the pipeline to the fitted model.
clf = model.named_steps["logisticregression"]
# argsort orders feature indices from most negative to most positive weight
order = np.argsort(clf.coef_[0])
print("\nFeatures most indicative of MALIGNANT:")
# the first four indices: the most negative weights
for i in order[:4]:
    print(f"  {data.feature_names[i]:25s} weight {clf.coef_[0][i]:+.2f}")
print("Features most indicative of BENIGN:")
# the last three indices: the most positive weights
for i in order[-3:]:
    print(f"  {data.feature_names[i]:25s} weight {clf.coef_[0][i]:+.2f}")
 
# ---- Check the sigmoid by hand for one patient ---------------------------
# Repeat exactly what the model does internally: scale the features with the
# training-set statistics, form z = w.x + b, then squash z into (0, 1)
# with the sigmoid. Large positive z -> probability near 1 (benign).
x_scaled = model.named_steps["standardscaler"].transform(X_test[:1])
# @ is the dot product: the weighted sum of the 30 scaled features
z = x_scaled @ clf.coef_[0] + clf.intercept_[0]
print(f"\nManual sigmoid for patient 0: {1 / (1 + np.exp(-z[0])):.3f} "
      f"(matches predict_proba)")

Test accuracy: 0.958
P(benign) = 0.996   actual: benign
P(benign) = 0.000   actual: malignant
P(benign) = 0.000   actual: malignant
P(benign) = 1.000   actual: benign
P(benign) = 0.003   actual: malignant

Features most indicative of MALIGNANT:
  radius error              weight -1.21
  worst texture             weight -1.11
  mean concave points       weight -1.11
  worst radius              weight -0.99
Features most indicative of BENIGN:
  symmetry error            weight +0.47
  fractal dimension error   weight +0.55
  compactness error         weight +0.70

Manual sigmoid for patient 0: 0.996 (matches predict_proba)


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Test accuracy: 0.958
P(benign) = 0.996   actual: benign
P(benign) = 0.000   actual: malignant
P(benign) = 0.000   actual: malignant
P(benign) = 1.000   actual: benign
P(benign) = 0.003   actual: malignant
 
Features most indicative of MALIGNANT:
  radius error              weight -1.21
  worst texture             weight -1.11
  mean concave points       weight -1.11
  worst radius              weight -0.99
Features most indicative of BENIGN:
  symmetry error            weight +0.47
  fractal dimension error   weight +0.55
  compactness error         weight +0.70
 
Manual sigmoid for patient 0: 0.996 (matches predict_proba)
```

**Investigate**

1. Remove StandardScaler from the pipeline. Does accuracy change? Are the weights still comparable with one another? Why?
2. In a cancer screening context, is it worse to miss a malignant tumour or to flag a benign one? Change the threshold so fewer malignant tumours are missed and measure the effect using the metrics in Section 3.11.
3. Try LogisticRegression(C=0.01) and C=100. C is the inverse of regularisation strength. What happens to the size of the weights?

## 4. L2_trees.py

**File:** `l2_trees.py`

Boosting also combines many trees but builds them in sequence rather than independently. The first tree makes a rough prediction. Each following tree, usually shallow, is trained specifically on what the ensemble so far still gets wrong (strictly, the gradient of the loss function) and nudges the combined prediction towards the right answer. Each tree's contribution is scaled down by a learning rate, so the model improves in many small steps rather than a few large ones (Friedman, 2001).

The two ensembles attack different weaknesses. Bagging averages many deep, overconfident trees to reduce variance. Boosting adds many shallow, weak trees to reduce bias, the systematic errors a simple model makes. Boosting is usually the more accurate of the two when well-tuned, but it is more sensitive to its settings: too many trees or too high a learning rate will overfit. On the small wine dataset, gradient boosting with 200 trees and the default learning rate averages 94% under cross-validation, between the single tree and the forest. On larger and harder datasets, tuned boosting usually comes out ahead.

Highly optimised implementations, XGBoost (Chen and Guestrin, 2016), LightGBM (Ke et al., 2017) and CatBoost, add refinements such as regularisation, efficient handling of missing values and fast training on large datasets. They dominate machine learning competitions on tabular data. A 2022 benchmark study found that tree-based ensembles still generally outperformed deep learning on typical medium-sized tabular datasets (Grinsztajn et al., 2022), which is why they remain the default choice for business data held in spreadsheets and databases.

Table: Single trees and tree ensembles compared

In one sentence: a random forest asks many independent trees and takes a vote, while gradient boosting builds each tree to fix the mistakes of the ones before it.

Example code: one tree, a forest and a boosted ensemble

The wine dataset records 13 chemical measurements for wines from three cultivars grown in the same region of Italy.

In [5]:
"""
Layer 2 example: decision trees, random forests and gradient boosting.
File: l2_trees.py
 
Purpose:
  Tree-based models are the workhorses of machine learning on tables of
  numbers. This program compares a single decision tree with two ENSEMBLES
  (many trees combined) to show why ensembles usually generalise better,
  and how a shallow tree can be read as human IF-THEN rules.
 
  Dataset: wine recognition (178 wines, 13 chemical measurements, 3 cultivars).
 
    Decision tree     : one tree of IF-THEN splits, easy to read, overfits
    Random forest     : many trees on random samples, votes averaged (bagging)
    Gradient boosting : trees built in sequence, each fixing earlier errors
 
How it works:
  1. Split the wines into training (70%) and test (30%) sets, stratified.
  2. Train four models and report training, test and 5-fold CV accuracy.
  3. Print the rules learned by the depth-2 tree.
  4. List the five features the random forest relied on most.
 
What to look for in the output:
  - The unlimited tree scores 1.000 on training data but only 0.888 in
    cross-validation: a perfect training score is a sign of overfitting.
  - The random forest has the best CV accuracy (0.967).
  - The depth-2 tree needs only two measurements (colour intensity and
    flavanoids) yet still reaches about 0.82 to 0.89.
  - Proline, colour intensity and flavanoids are the most important features.
 
Requires: pip install scikit-learn
"""
 
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
 
# ---- Load and split the data ---------------------------------------------
# Trees split on one feature at a time, so (unlike k-NN or SVM) they do not
# need the features to be scaled.
X, y = load_wine(return_X_y=True)
names = load_wine().feature_names
# stratify=y keeps the proportion of each cultivar equal in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)
 
# ---- Four models, from simplest to most powerful --------------------------
# random_state fixes the randomness inside each model so results repeat.
models = {
    # No depth limit: the tree keeps splitting until every leaf is pure
    "Tree (unlimited depth)": DecisionTreeClassifier(random_state=0),
    # max_depth is a HYPERPARAMETER that limits complexity (regularisation)
    "Tree (max_depth=2)":     DecisionTreeClassifier(max_depth=2, random_state=0),
    # 300 trees, each trained on a BOOTSTRAP sample: rows drawn at random WITH
    # replacement, so each tree sees a slightly different dataset. Each split
    # also considers a random subset of features. Averaging many different,
    # overfitted trees cancels out much of their individual noise (variance).
    "Random forest":          RandomForestClassifier(n_estimators=300,
                                                     random_state=0),
    # 200 shallow trees added one after another; each new tree is fitted to
    # the errors of the ensemble so far, and learning_rate (default 0.1)
    # shrinks each tree's contribution so no single tree dominates.
    "Gradient boosting":      GradientBoostingClassifier(n_estimators=200,
                                                         random_state=0),
}
 
# ---- Train and compare -----------------------------------------------------
# Train  : accuracy on data the model has seen (optimistic)
# Test   : accuracy on the held-out 30% (one estimate of generalisation)
# CV     : average over 5 train/test splits (a more stable estimate)
print(f"{'Model':25s} {'Train':>6s} {'Test':>6s} {'5-fold CV':>10s}")
for name, m in models.items():
    m.fit(X_train, y_train)
    # cross_val_score retrains the model on 5 different splits of the data
    cv = cross_val_score(m, X, y, cv=5).mean()
    print(f"{name:25s} {m.score(X_train, y_train):6.3f} "
          f"{m.score(X_test, y_test):6.3f} {cv:10.3f}")
 
# ---- The shallow tree is readable by a human ------------------------------
# export_text prints the tree as nested IF-THEN rules; each leaf shows the
# class predicted for wines that reach it. This interpretability is the
# main advantage of a single small tree over an ensemble.
print("\nRules learned by the depth-2 tree:")
print(export_text(models["Tree (max_depth=2)"], feature_names=names))
 
# ---- Feature importance: which measurements mattered most? ---------------
# Impurity-based importance: total reduction in Gini impurity per feature.
# Gini impurity measures how mixed the classes are in a node (0 = pure).
# The importances are averaged over all 300 trees and sum to 1.
rf = models["Random forest"]
# pair each importance with its feature name, sort largest first, keep 5
top = sorted(zip(rf.feature_importances_, names), reverse=True)[:5]
print("Top 5 features in the random forest:")
for imp, n in top:
    print(f"  {n:30s} {imp:.3f}")

Model                      Train   Test  5-fold CV


Tree (unlimited depth)     1.000  0.944      0.888
Tree (max_depth=2)         0.919  0.889      0.821


Random forest              1.000  1.000      0.967


Gradient boosting          1.000  0.981      0.939

Rules learned by the depth-2 tree:
|--- color_intensity <= 3.46
|   |--- class: 1
|--- color_intensity >  3.46
|   |--- flavanoids <= 1.58
|   |   |--- class: 2
|   |--- flavanoids >  1.58
|   |   |--- class: 0

Top 5 features in the random forest:
  proline                        0.181
  color_intensity                0.170
  flavanoids                     0.145
  od280/od315_of_diluted_wines   0.122
  alcohol                        0.118


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Model                      Train   Test  5-fold CV
Tree (unlimited depth)     1.000  0.944      0.888
Tree (max_depth=2)         0.919  0.889      0.821
Random forest              1.000  1.000      0.967
Gradient boosting          1.000  0.981      0.939
 
Rules learned by the depth-2 tree:
|--- color_intensity <= 3.46
|   |--- class: 1
|--- color_intensity >  3.46
|   |--- flavanoids <= 1.58
|   |   |--- class: 2
|   |--- flavanoids >  1.58
|   |   |--- class: 0
 
Top 5 features in the random forest:
  proline                        0.181
  color_intensity                0.170
  flavanoids                     0.145
  od280/od315_of_diluted_wines   0.122
  alcohol                        0.118
```

**What the output shows**

Notice that the unlimited tree scores 1.000 on training data, but its cross-validated score is much lower: clear evidence of overfitting. The forest and boosted model generalise far better. The printed depth-2 tree is exactly the tree shown in the figure at the start of this section.

**Investigate**

1. Using the random forest, record 5-fold CV accuracy for n_estimators of 1, 5, 25, 100 and 500. At what point do extra trees stop helping?
2. Impurity-based importance can overstate features with many unique values. Research sklearn.inspection.permutation_importance and compare its ranking.
3. Install XGBoost (pip install xgboost) and add XGBClassifier to the comparison.

## 5. K-NN from scratch and SVM kernels

**File:** `l2_knn_svm.py`

In [6]:
"""
Layer 2 example: k-nearest neighbours (from scratch) and support vector machines.
File: l2_knn_svm.py
 
Purpose:
  k-NN and SVMs are two classic ways to draw a curved decision boundary. The
  program builds k-NN from scratch to show there is no hidden magic, checks it
  against scikit-learn, and then shows how an SVM's kernel and its C
  hyperparameter control the shape and flexibility of the boundary.
 
  Dataset: 'two moons', a curved shape that no straight line can separate.
 
  k-NN  : no training at all. To classify a point, find the k closest
          training points and take a majority vote. A 'lazy' learner.
  SVM   : finds the boundary with the WIDEST MARGIN between classes.
          The 'kernel trick' lets it draw curved boundaries by implicitly
          mapping data into a higher-dimensional space.
 
How it works:
  1. Generate 400 noisy two-moons points and split 70/30 into train/test.
  2. Classify the test points with our own k-NN for several k, and compare
     with scikit-learn's KNeighborsClassifier.
  3. Train SVMs with linear, polynomial and RBF kernels.
  4. Vary C for the RBF SVM and compare training and test accuracy.
 
What to look for in the output:
  - Our k-NN matches scikit-learn exactly for every k.
  - k=15 does best (0.925); k=1 overfits and k=101 underfits.
  - The linear kernel is worst (0.850): a straight line cannot fit moons.
  - Raising C pushes training accuracy up (0.879 -> 0.986) while test
    accuracy does not keep improving: the model starts fitting noise.
 
Requires: pip install numpy scikit-learn
"""
 
import numpy as np
from collections import Counter
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
 
# ---- Data ------------------------------------------------------------------
# noise=0.25 blurs the moons so they overlap slightly, as real data would.
# Both features already have similar ranges here, so we skip scaling. On real
# data you MUST standardise before k-NN or SVM: both rely on distances, and a
# feature measured in large units would otherwise dominate the distance.
X, y = make_moons(n_samples=400, noise=0.25, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=3)
 
 
# ---- k-NN from scratch -----------------------------------------------------
# 'Training' k-NN is just storing the data; all the work happens at
# prediction time, which is why it becomes slow on large datasets.
def knn_predict(x, k):
    # Euclidean distance from x to EVERY training point: sqrt(sum((a-b)^2))
    dists = np.sqrt(((X_train - x) ** 2).sum(axis=1))
    nearest = np.argsort(dists)[:k]            # indices of the k closest
    votes = Counter(y_train[nearest])          # count labels among them
    return votes.most_common(1)[0][0]          # majority wins
 
 
# k is the key hyperparameter: how many neighbours get a vote.
print("k-NN from scratch vs scikit-learn (test accuracy):")
for k in (1, 5, 15, 101):
    # fraction of test points where our prediction equals the true label
    ours = np.mean([knn_predict(x, k) == t for x, t in zip(X_test, y_test)])
    sk = (KNeighborsClassifier(n_neighbors=k)
          .fit(X_train, y_train).score(X_test, y_test))
    print(f"  k={k:3d}  ours {ours:.3f}  sklearn {sk:.3f}")
# Small k follows noise (overfits); very large k blurs the boundary (underfits).
 
# ---- SVM with different kernels ------------------------------------------
# A kernel measures similarity between two points:
#   linear : a straight boundary only
#   poly   : polynomial curves (degree 3 here)
#   rbf    : 'radial basis function', similarity falls with distance; gamma
#            sets how quickly, so larger gamma -> more wiggly boundary
# C=1.0 is the default penalty for points on the wrong side of the margin.
print("\nSupport vector machines (test accuracy):")
for kernel, params in [("linear", {}), ("poly", {"degree": 3}),
                       ("rbf", {"gamma": 1.0})]:
    # **params passes the kernel-specific hyperparameter (degree or gamma)
    svm = SVC(kernel=kernel, C=1.0, **params).fit(X_train, y_train)
    # n_support_ counts support vectors per class; sum gives the total
    print(f"  {kernel:6s} kernel  accuracy {svm.score(X_test, y_test):.3f}  "
          f"support vectors {svm.n_support_.sum()}")
# Only the SUPPORT VECTORS (points on or inside the margin) define the boundary.
 
# ---- The C hyperparameter ------------------------------------------------
# Small C: a wide, 'soft' margin that tolerates errors (simpler model).
# Large C: every training error is punished heavily, so the boundary bends
# to fit individual points (risk of overfitting). Compare train vs test.
print("\nEffect of C on the RBF SVM (C controls tolerance of misclassified points):")
for C in (0.01, 1, 100, 10000):
    svm = SVC(kernel="rbf", C=C, gamma=1.0).fit(X_train, y_train)
    print(f"  C={C:<6}  train {svm.score(X_train, y_train):.3f}  "
          f"test {svm.score(X_test, y_test):.3f}")

k-NN from scratch vs scikit-learn (test accuracy):
  k=  1  ours 0.900  sklearn 0.900
  k=  5  ours 0.908  sklearn 0.908
  k= 15  ours 0.925  sklearn 0.925
  k=101  ours 0.875  sklearn 0.875

Support vector machines (test accuracy):
  linear kernel  accuracy 0.850  support vectors 95
  poly   kernel  accuracy 0.908  support vectors 107
  rbf    kernel  accuracy 0.925  support vectors 73

Effect of C on the RBF SVM (C controls tolerance of misclassified points):
  C=0.01    train 0.879  test 0.858
  C=1       train 0.932  test 0.925
  C=100     train 0.961  test 0.908


  C=10000   train 0.986  test 0.925


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
k-NN from scratch vs scikit-learn (test accuracy):
  k=  1  ours 0.900  sklearn 0.900
  k=  5  ours 0.908  sklearn 0.908
  k= 15  ours 0.925  sklearn 0.925
  k=101  ours 0.875  sklearn 0.875
 
Support vector machines (test accuracy):
  linear kernel  accuracy 0.850  support vectors 95
  poly   kernel  accuracy 0.908  support vectors 107
  rbf    kernel  accuracy 0.925  support vectors 73
 
Effect of C on the RBF SVM (C controls tolerance of misclassified points):
  C=0.01    train 0.879  test 0.858
  C=1       train 0.932  test 0.925
  C=100     train 0.961  test 0.908
  C=10000   train 0.986  test 0.925
```

**Investigate**

1. Your from-scratch k-NN matches scikit-learn exactly. Time both on 10,000 test points using the time module. scikit-learn uses tree structures (KD-trees and ball trees) to avoid comparing against every point.
2. Try gamma values of 0.1, 1 and 20 for the RBF SVM and relate the results to underfitting and overfitting.
3. Why does the linear kernel perform worst on this dataset? Refer to the decision boundaries figure.

## 6. A spam filter

**File:** `l2_naive_bayes.py`

In [7]:
"""
Layer 2 example: a naive Bayes spam filter.
File: l2_naive_bayes.py
 
Purpose:
  Naive Bayes applies Bayes' theorem to text, turning word counts into the
  probability that a message is spam. It trains in a single pass, works with
  very little data and was the basis of early email spam filters, which makes
  it a clear first example of probabilistic classification.
 
  Bayes' theorem:  P(spam | words) is proportional to P(words | spam) * P(spam)
  'Naive' because it assumes each word appears independently of the others,
  which is false, yet the method works remarkably well for text.
 
How it works:
  1. Start from ten labelled messages (five spam, five ham, i.e. not spam).
  2. Convert each message into a 'bag of words': a vector of word counts.
  3. Fit a multinomial naive Bayes model with Laplace smoothing.
  4. Rank words by how strongly they point to spam or ham.
  5. Score three new messages.
 
What to look for in the output:
  - After removing common English 'stop words', 36 distinct words remain.
  - Spam words (win, cash, click, free) and ham words (meeting, review,
    team) match intuition, learned from only ten examples.
  - 'Free lunch at the team meeting' gets only P(spam)=0.12: the ham words
    outweigh 'free'. Evidence is combined, not judged word by word.
 
Requires: pip install numpy scikit-learn
"""
 
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
 
# ---- A tiny labelled training set ------------------------------------------
# Real filters learn from millions of messages; ten are enough to show the idea.
train = [
    ("Win a FREE prize now, click the link", "spam"),
    ("Claim your cash reward today, limited offer", "spam"),
    ("URGENT: your account wins a free holiday", "spam"),
    ("Cheap loans, click now to claim cash", "spam"),
    ("Exclusive offer, win money fast", "spam"),
    ("Are we still meeting for lunch tomorrow?", "ham"),
    ("Please review the attached project report", "ham"),
    ("Can you send me the lecture notes from today", "ham"),
    ("The team meeting has moved to Thursday", "ham"),
    ("Thanks for your help with the code review", "ham"),
]
# split the (text, label) pairs into two parallel tuples
texts, labels = zip(*train)
 
# ---- Turn text into numbers: a 'bag of words' --------------------------------
# Each column is a word; each value is how many times it occurs in a message.
# Word order is thrown away: only counts matter.
# lowercase=True treats 'FREE' and 'free' as the same word.
# stop_words="english" drops very common words ('the', 'your', 'to') that
# carry little information about spam.
vec = CountVectorizer(lowercase=True, stop_words="english")
# fit_transform learns the vocabulary AND converts the texts to count vectors
X = vec.fit_transform(texts)
print("Vocabulary size:", len(vec.vocabulary_))
 
# ---- Train the classifier -------------------------------------------------
# Training just counts how often each word appears in spam and in ham, and
# turns the counts into probabilities P(word | class). The class priors
# P(spam) and P(ham) come from the label frequencies (here 50/50).
# alpha=1 is LAPLACE SMOOTHING: add 1 to every count so an unseen word
# does not give a probability of exactly zero and wipe out all other evidence.
# (Probabilities are multiplied together, so a single zero would make the
# whole product zero, however strong the other words are.)
nb = MultinomialNB(alpha=1.0).fit(X, labels)
 
# ---- Which words are most 'spammy'? -------------------------------------------
words = np.array(vec.get_feature_names_out())
# classes_ is sorted alphabetically (ham, spam), so look up spam's row index
spam_i = list(nb.classes_).index("spam")
# feature_log_prob_ stores log P(word | class). Logs are used because
# multiplying many small probabilities would underflow to zero; adding
# logs is equivalent and numerically safe.
# log P(word|spam) - log P(word|ham): positive means the word points to spam
ratio = nb.feature_log_prob_[spam_i] - nb.feature_log_prob_[1 - spam_i]
# argsort sorts from most ham-like to most spam-like; [::-1] reverses
print("Strongest spam words:", list(words[np.argsort(ratio)[-6:]][::-1]))
print("Strongest ham words: ", list(words[np.argsort(ratio)[:6]]))
 
# ---- Classify new messages -------------------------------------------------
# New text must go through the SAME vectoriser (transform, not fit_transform)
# so each word maps to the same column. Words not in the vocabulary, such as
# 'Friday', are simply ignored.
tests = ["Click here to claim your free cash prize",
         "Can we move the project meeting to Friday",
         "Free lunch at the team meeting today"]
for t, p in zip(tests, nb.predict_proba(vec.transform(tests))):
    print(f"P(spam)={p[spam_i]:.2f}  {t}")

Vocabulary size: 36
Strongest spam words: ['win', 'cash', 'click', 'offer', 'claim', 'free']
Strongest ham words:  ['review', 'meeting', 'attached', 'code', 'lunch', 'team']
P(spam)=0.99  Click here to claim your free cash prize
P(spam)=0.12  Can we move the project meeting to Friday
P(spam)=0.12  Free lunch at the team meeting today


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Vocabulary size: 36
Strongest spam words: ['win', 'cash', 'click', 'offer', 'claim', 'free']
Strongest ham words:  ['review', 'meeting', 'attached', 'code', 'lunch', 'team']
P(spam)=0.99  Click here to claim your free cash prize
P(spam)=0.12  Can we move the project meeting to Friday
P(spam)=0.12  Free lunch at the team meeting today
```

**Investigate**

1. The third test message contains "free" but is classified as legitimate. Explain why, using the word lists printed above it.
2. Replace CountVectorizer with TfidfVectorizer, which down-weights words common across all messages. Does anything change?
3. Set alpha to a very small value such as 1e-10 and test a message containing a word that only appears in ham. What happens and why?

## 7. K-means from scratch and PCA

**File:** `l2_unsupervised.py`

In [8]:
"""
Layer 2 example: unsupervised learning with k-means and PCA.
File: l2_unsupervised.py
 
Purpose:
  Unsupervised learning finds structure in data WITHOUT labels. k-means
  groups similar points into clusters; PCA compresses many features into a
  few. Both are used to explore data, segment customers and simplify inputs
  before other models are trained.
 
  No labels are used for training. We only look at the true labels AFTERWARDS
  to see whether the discovered structure matches reality.
 
  k-means (Lloyd's algorithm):
    1. place k centroids at random data points
    2. ASSIGN each point to its nearest centroid
    3. UPDATE each centroid to the mean of its assigned points
    4. repeat 2 and 3 until nothing changes
  PCA (principal component analysis) finds new axes that capture as much
  variance as possible, compressing many features into a few.
 
How it works:
  1. Generate 150 points around three known centres.
  2. Run our own k-means five times from different random starts.
  3. Use the elbow method with scikit-learn's KMeans to choose k.
  4. Apply PCA to 8 x 8 images of handwritten digits (64 pixels each).
  5. Cluster the digits with k-means and compare the clusters with the
     true digit labels.
 
What to look for in the output:
  - Run 0 gets stuck at inertia 1142.7; the other runs reach 302.3. Random
    starts matter, so keep the best of several.
  - The recovered centroids are close to the true centres.
  - Inertia drops sharply up to k=3, then only slowly: the elbow is at 3.
  - 20 PCA components keep 89% of the variance, and clustering on them
    works as well as on all 64 pixels (ARI about 0.67).
 
Requires: pip install numpy scikit-learn
"""
 
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
 
# One seeded random generator, so the 'random' starts repeat on every run.
rng = np.random.default_rng(0)
 
# ---- k-means from scratch on simple 2D data --------------------------------
# Three blobs of 50 points each, scattered (standard deviation 1) around
# known centres, so we can check what k-means finds.
centres_true = np.array([[2, 2], [8, 3], [5, 8]])
pts = np.vstack([c + rng.normal(0, 1.0, (50, 2)) for c in centres_true])
 
 
# INERTIA is the sum of squared distances from each point to its assigned
# centroid. Lower inertia means tighter clusters; k-means tries to minimise it.
def kmeans(data, k, iters=20, verbose=False):
    # start from k distinct data points chosen at random
    centroids = data[rng.choice(len(data), k, replace=False)]
    for i in range(iters):
        # distance of every point to every centroid -> shape (n_points, k)
        # (None adds an axis so NumPy broadcasting compares all pairs at once)
        d = np.linalg.norm(data[:, None, :] - centroids[None, :, :], axis=2)
        labels = d.argmin(axis=1)                       # ASSIGN step
        # UPDATE step: each centroid moves to the mean of its points
        new = np.array([data[labels == j].mean(axis=0) for j in range(k)])
        inertia = ((data - new[labels]) ** 2).sum()     # within-cluster spread
        if verbose:
            print(f"    iteration {i}: inertia {inertia:8.1f}")
        if np.allclose(new, centroids):                 # converged
            break
        centroids = new
    return centroids, inertia
 
 
# k-means only finds a LOCAL minimum: a poor random start can merge two real
# clusters and split another. The standard fix is several restarts, keeping
# the result with the lowest inertia (scikit-learn's n_init parameter).
print("k-means from scratch, 5 random restarts:")
runs = []
for run in range(5):
    # print the iteration-by-iteration inertia for the first run only
    centroids, inertia = kmeans(pts, 3, verbose=(run == 0))
    print(f"  run {run}: final inertia {inertia:8.1f}")
    runs.append((inertia, centroids))
# keep the centroids from the run with the smallest inertia
found = min(runs, key=lambda r: r[0])[1]
print("Best run kept.")
# sort both sets by x coordinate so they line up for comparison
print("Centroids found:\n", np.round(found[np.argsort(found[:, 0])], 2))
print("True centres:\n", centres_true[np.argsort(centres_true[:, 0])])
 
# ---- Choosing k with the elbow method --------------------------------------
# Inertia always falls as k grows (with k = n points it reaches 0), so we
# cannot just pick the k with the lowest inertia. Instead we look for the
# point of diminishing returns.
# n_init=10: run 10 random starts and keep the best, as we did by hand above.
print("\nElbow method (inertia for each k):")
for k in range(1, 7):
    print(f"  k={k}  inertia "
          f"{KMeans(n_clusters=k, n_init=10, random_state=0).fit(pts).inertia_:8.1f}")
# The 'elbow' where improvement suddenly slows suggests a good k (here 3).
 
# ---- PCA on handwritten digits (64 pixel features) -------------------------
# Each image is 8 x 8 = 64 grey levels. Neighbouring pixels are strongly
# correlated, so far fewer than 64 numbers are needed to describe an image.
X, y = load_digits(return_X_y=True)
# PCA() with no argument keeps all 64 components so we can inspect them
pca = PCA().fit(X)
# explained_variance_ratio_: share of total variance captured by each new
# axis, largest first. The cumulative sum shows how much k axes keep.
cum = np.cumsum(pca.explained_variance_ratio_)
for n in (2, 10, 20, 40):
    print(f"PCA: {n:2d} components keep {cum[n-1]*100:5.1f}% "
          f"of the variance of 64 pixels")
 
# Cluster the digits WITHOUT labels, then compare with the real digits.
# Adjusted Rand index: 0 = random agreement, 1 = perfect agreement.
# (ARI ignores cluster numbering: it only asks whether pairs of images that
# share a digit also share a cluster.)
# PCA(20).fit_transform reduces each image from 64 numbers to 20.
for label, data in [("raw 64 pixels", X),
                    ("20 PCA components", PCA(20).fit_transform(X))]:
    # 10 clusters, one hoped-for cluster per digit 0 to 9
    clusters = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(data)
    print(f"k-means on {label:18s}: agreement with true digits (ARI) "
          f"{adjusted_rand_score(y, clusters):.3f}")

k-means from scratch, 5 random restarts:
    iteration 0: inertia   1143.5
    iteration 1: inertia   1142.7
    iteration 2: inertia   1142.7
  run 0: final inertia   1142.7
  run 1: final inertia    302.3
  run 2: final inertia    302.3
  run 3: final inertia    302.3
  run 4: final inertia    302.3
Best run kept.
Centroids found:
 [[1.99 2.18]
 [4.65 8.08]
 [7.87 3.03]]
True centres:
 [[2 2]
 [5 8]
 [8 3]]

Elbow method (inertia for each k):


  k=1  inertia   2187.6


  k=2  inertia   1186.3


  k=3  inertia    302.3


  k=4  inertia    257.4


  k=5  inertia    219.7


  k=6  inertia    184.9
PCA:  2 components keep  28.5% of the variance of 64 pixels
PCA: 10 components keep  73.8% of the variance of 64 pixels
PCA: 20 components keep  89.4% of the variance of 64 pixels
PCA: 40 components keep  98.8% of the variance of 64 pixels


k-means on raw 64 pixels     : agreement with true digits (ARI) 0.666


k-means on 20 PCA components : agreement with true digits (ARI) 0.668


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
k-means from scratch, 5 random restarts:
    iteration 0: inertia   1143.5
    iteration 1: inertia   1142.7
    iteration 2: inertia   1142.7
  run 0: final inertia   1142.7
  run 1: final inertia    302.3
  run 2: final inertia    302.3
  run 3: final inertia    302.3
  run 4: final inertia    302.3
Best run kept.
Centroids found:
 [[1.99 2.18]
 [4.65 8.08]
 [7.87 3.03]]
True centres:
 [[2 2]
 [5 8]
 [8 3]]
 
Elbow method (inertia for each k):
  k=1  inertia   2187.6
  k=2  inertia   1186.3
  k=3  inertia    302.3
  k=4  inertia    257.4
  k=5  inertia    219.7
  k=6  inertia    184.9
PCA:  2 components keep  28.5% of the variance of 64 pixels
PCA: 10 components keep  73.8% of the variance of 64 pixels
PCA: 20 components keep  89.4% of the variance of 64 pixels
PCA: 40 components keep  98.8% of the variance of 64 pixels
k-means on raw 64 pixels     : agreement with true digits (ARI) 0.666
k-means on 20 PCA components : agreement with true digits (ARI) 0.668
```

**What the output shows**

The output demonstrates the local minimum problem directly: run 0 converged to a poor solution with inertia 1142.7, while other starts found the correct clusters with inertia 302.3. The elbow method also shows a sharp bend at k = 3, the true number of clusters.

**Investigate**

1. Implement k-means++ initialisation: choose the first centroid at random, then choose each subsequent centroid with probability proportional to its squared distance from the nearest existing centroid. Do all runs now reach 302.3?
2. Use sklearn.metrics.silhouette_score to choose k for the digits data. Does it suggest 10?
3. Create a scatter plot of the digits using sklearn.manifold.TSNE and compare it with the PCA figure.

## 8. Overfitting and cross-validation

**File:** `l2_overfitting.py`

The example uses the diabetes dataset supplied with scikit-learn: ten baseline measurements for 442 patients and a measure of disease progression one year later.

In [9]:
"""
Layer 2 example: overfitting, cross-validation and hyperparameter tuning.
File: l2_overfitting.py
 
Purpose:
  A model that is too flexible memorises noise in its training data and then
  fails on new data: OVERFITTING. This program makes the problem visible in
  numbers, shows how regularisation tames it, and shows how cross-validation
  gives a trustworthy way to choose hyperparameters without touching a test set.
 
  Part 1 shows the bias-variance trade-off numerically.
  Part 2 shows k-fold cross-validation and GridSearchCV.
 
How it works:
  1. Sample 20 noisy training points from a sine curve, plus 500 test points.
  2. Fit polynomials of degree 1 to 15 and compare training and test error.
  3. Refit the degree-15 polynomial with Ridge regularisation of varying strength.
  4. Run 5-fold cross-validation of Ridge regression on the diabetes dataset.
  5. Let GridSearchCV pick the best Ridge alpha by cross-validation.
 
What to look for in the output:
  - Training MSE falls steadily with degree, but test MSE is lowest at
    degree 3 and explodes to about 258,000 at degree 15.
  - Ridge brings the degree-15 test MSE down to about 0.21, but too much
    regularisation (alpha=1) makes it worse again (underfitting).
  - Fold scores vary from 0.34 to 0.50: one split alone would be misleading.
  - Grid search picks alpha = 0.001 (mean CV R2 0.489).
 
Requires: pip install numpy scikit-learn
"""
 
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.datasets import load_diabetes
 
rng = np.random.default_rng(1)
 
 
# ---- Part 1: polynomial models of increasing complexity -------------------
# The true pattern is a sine wave; normal noise with standard deviation 0.3
# stands in for measurement error that no model should try to learn.
def make_data(n):
    x = rng.uniform(0, 1, n)
    # reshape(-1, 1) turns x into a one-column matrix, as scikit-learn expects
    return x.reshape(-1, 1), np.sin(2 * np.pi * x) + rng.normal(0, 0.3, n)
 
 
X_train, y_train = make_data(20)          # small training set
X_test, y_test = make_data(500)           # large unseen test set
 
# PolynomialFeatures(d) adds columns x^2, x^3, ... x^d, so a LINEAR model
# can fit a curve. Higher degree = more weights = a more flexible model.
# MSE (mean squared error) is the average squared prediction error.
print("Degree   train MSE   test MSE")
for degree in (1, 3, 5, 9, 15):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    tr = mean_squared_error(y_train, model.predict(X_train))
    te = mean_squared_error(y_test, model.predict(X_test))
    print(f"  {degree:2d}     {tr:8.3f}   {te:10.3f}")
# Training error ALWAYS falls as complexity rises.
# Test error falls, then rises sharply: that rise is overfitting.
# Degree 1 underfits (high BIAS: a line cannot follow a sine wave).
# Degree 15 overfits (high VARIANCE: 16 weights chasing 20 noisy points).
 
# ---- Regularisation -------------------------------------------------------
# Ridge adds alpha * (sum of squared weights) to the loss. Huge weights are
# what let a polynomial swing wildly between points, so penalising them
# smooths the curve while keeping all 15 degrees available.
# alpha is a hyperparameter: tiny alpha ~ no penalty, large alpha ~ too stiff.
print("\nSame degree-15 model with Ridge regularisation (penalises large weights):")
for alpha in (1e-8, 1e-4, 1e-2, 1):
    model = make_pipeline(PolynomialFeatures(15), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    print(f"  alpha={alpha:<7}  test MSE "
          f"{mean_squared_error(y_test, model.predict(X_test)):.3f}")
 
# ---- Part 2: cross-validation on a real dataset ---------------------------
# k-fold cross-validation: cut the data into 5 folds; train on 4, score on
# the 5th, and rotate so every row is used for scoring exactly once. The
# average is a steadier estimate than a single train/test split.
X, y = load_diabetes(return_X_y=True)       # 442 patients, 10 features
# shuffle=True mixes the rows first in case the file is sorted in some way
kf = KFold(n_splits=5, shuffle=True, random_state=0)
# R2 (R-squared): 1 = perfect, 0 = no better than predicting the mean
scores = cross_val_score(Ridge(alpha=1.0), X, y, cv=kf, scoring="r2")
print("\n5-fold cross-validation R2 per fold:", np.round(scores, 3))
# the standard deviation shows how much the score depends on the split
print(f"Mean {scores.mean():.3f}  standard deviation {scores.std():.3f}")
 
# ---- Hyperparameter tuning ------------------------------------------------
# GridSearchCV tries every hyperparameter value with cross-validation
# (6 alphas x 5 folds = 30 fits), then refits the best one on all the data.
# Choosing alpha with CV, not a test set, keeps any final test score honest.
grid = GridSearchCV(Ridge(), {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]},
                    cv=kf, scoring="r2")
grid.fit(X, y)
print("Best alpha found by grid search:", grid.best_params_["alpha"],
      f"(mean CV R2 {grid.best_score_:.3f})")

Degree   train MSE   test MSE
   1        0.164        0.347
   3        0.086        0.145
   5        0.075        0.164
   9        0.061        0.368
  15        0.035   258438.009

Same degree-15 model with Ridge regularisation (penalises large weights):
  alpha=1e-08    test MSE 0.435
  alpha=0.0001   test MSE 0.223
  alpha=0.01     test MSE 0.209
  alpha=1        test MSE 0.345

5-fold cross-validation R2 per fold: [0.341 0.41  0.447 0.407 0.497]
Mean 0.420  standard deviation 0.051


Best alpha found by grid search: 0.001 (mean CV R2 0.489)


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Degree   train MSE   test MSE
   1        0.164        0.347
   3        0.086        0.145
   5        0.075        0.164
   9        0.061        0.368
  15        0.035   258438.009
 
Same degree-15 model with Ridge regularisation (penalises large weights):
  alpha=1e-08    test MSE 0.435
  alpha=0.0001   test MSE 0.223
  alpha=0.01     test MSE 0.209
  alpha=1        test MSE 0.345
 
5-fold cross-validation R2 per fold: [0.341 0.41  0.447 0.407 0.497]
Mean 0.420  standard deviation 0.051
Best alpha found by grid search: 0.001 (mean CV R2 0.489)
```

**What the output shows**

The degree-15 model achieves the lowest training error of all, yet its test error is enormous: an extreme and unmistakable case of overfitting. Ridge regularisation brings the same model back under control. Note that grid search chose the smallest alpha offered; when the best value is at the edge of the search range, the range should be extended.

**Investigate**

1. Increase the training set from 20 to 200 points. What happens to the test error of the degree-15 model? Explain using the concept of variance.
2. Extend the alpha grid to include 0.00001 and 0.0001. Does the best value change?
3. Replace GridSearchCV with RandomizedSearchCV over a continuous range using scipy.stats.loguniform.

## 9. The accuracy trap

**File:** `l2_evaluation.py`

In [10]:
"""
Layer 2 example: why accuracy misleads, and how to evaluate properly.
File: l2_evaluation.py
 
Purpose:
  When one class is rare, accuracy is almost meaningless: a model that never
  predicts the rare class still scores highly. This program uses simulated
  card fraud to introduce the confusion matrix, precision, recall, F1, the
  decision threshold and ROC AUC, the measures used for real imbalanced
  problems such as fraud, disease screening and fault detection.
 
  We simulate card fraud: only about 2% of transactions are fraudulent.
 
How it works:
  1. Generate 20,000 synthetic transactions, about 2 to 3% fraudulent.
  2. Evaluate a 'dummy' model that always predicts genuine.
  3. Train a random forest and evaluate it at the default threshold of 0.5.
  4. Move the threshold and watch precision and recall trade off.
  5. Summarise ranking quality over all thresholds with ROC AUC.
 
What to look for in the output:
  - The dummy model scores 97.5% accuracy while catching no fraud at all.
  - At threshold 0.5 the forest is 95% precise but finds only 12.7% of fraud.
  - Lowering the threshold to 0.1 raises recall to 58.7%, at the cost of
    precision falling to 47.1% (more false alarms).
  - ROC AUC of 0.860 shows the model ranks fraud well above chance.
 
Requires: pip install numpy scikit-learn
"""
 
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)
 
# ---- Simulated transactions -----------------------------------------------
# 12 features, of which only 6 carry real information about fraud.
# weights=[0.98] makes class 0 (genuine) 98% of the data
# flip_y=0.01 randomly flips 1% of labels, like real mislabelled records
# (this is why the final fraud rate comes out nearer 2.5% than 2%).
X, y = make_classification(n_samples=20000, n_features=12, n_informative=6,
                           weights=[0.98], flip_y=0.01, random_state=7)
# stratify=y is essential with rare classes: without it, the test set could
# by chance contain far more or fewer fraud cases than the real rate.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=7)
print(f"Fraud rate in test set: {y_te.mean()*100:.1f}%\n")
 
 
# ---- Evaluation helper -------------------------------------------------------
# For fraud, 'positive' (1) means fraud:
#   TP = fraud caught            FN = fraud missed (money lost)
#   FP = genuine flagged         TN = genuine left alone
#   precision = TP / (TP + FP): of the transactions we flag, how many are fraud?
#   recall    = TP / (TP + FN): of all the fraud, how much do we catch?
#   F1        = harmonic mean of precision and recall (low if either is low)
def report(name, y_true, y_pred):
    # confusion_matrix returns [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"{name}")
    print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    # zero_division=0 avoids a warning when nothing is flagged (TP + FP = 0)
    print(f"  accuracy {accuracy_score(y_true, y_pred):.3f}  "
          f"precision {precision_score(y_true, y_pred, zero_division=0):.3f}  "
          f"recall {recall_score(y_true, y_pred):.3f}  "
          f"F1 {f1_score(y_true, y_pred, zero_division=0):.3f}")
 
 
# ---- A 'model' that always says genuine ------------------------------------
# A baseline: any real model must beat this to be worth using.
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
report("Always predict 'genuine'", y_te, dummy.predict(X_te))
# Very high accuracy, yet it catches ZERO fraud: recall = 0.
 
# ---- A real model -----------------------------------------------------------
# class_weight="balanced" weights errors on the rare class more heavily, so
# the forest does not simply learn to ignore fraud.
# n_jobs=-1 trains the 200 trees in parallel on all CPU cores.
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                            random_state=7, n_jobs=-1)
rf.fit(X_tr, y_tr)
proba = rf.predict_proba(X_te)[:, 1]           # probability of fraud
# (proba >= 0.5) gives True/False; astype(int) turns it into 1/0 labels
report("\nRandom forest, threshold 0.5", y_te, (proba >= 0.5).astype(int))
 
# ---- Moving the decision threshold trades precision for recall -------------
# A lower threshold flags more transactions: more fraud caught (recall up)
# but more genuine customers blocked (precision down). The right threshold
# is a business decision: the cost of missed fraud vs the cost of false alarms.
print("\nThreshold  precision  recall   flagged")
for t in (0.1, 0.2, 0.3, 0.5, 0.7):
    pred = (proba >= t).astype(int)
    # pred.sum() = number of transactions flagged as fraud
    print(f"   {t:.1f}      {precision_score(y_te, pred, zero_division=0):.3f}    "
          f"{recall_score(y_te, pred):.3f}   {pred.sum():5d}")
 
# ROC AUC measures ranking quality across ALL thresholds:
# the probability a random fraud case scores higher than a random genuine one.
# It needs the probabilities (proba), not the 0/1 predictions.
print(f"\nROC AUC: {roc_auc_score(y_te, proba):.3f}   "
      f"(0.5 = random guessing, 1.0 = perfect)")

Fraud rate in test set: 2.5%

Always predict 'genuine'
  TN=5850  FP=0  FN=150  TP=0
  accuracy 0.975  precision 0.000  recall 0.000  F1 0.000



Random forest, threshold 0.5
  TN=5849  FP=1  FN=131  TP=19
  accuracy 0.978  precision 0.950  recall 0.127  F1 0.224

Threshold  precision  recall   flagged
   0.1      0.471    0.587     187
   0.2      0.790    0.527     100
   0.3      0.857    0.360      63
   0.5      0.950    0.127      20
   0.7      1.000    0.027       4

ROC AUC: 0.860   (0.5 = random guessing, 1.0 = perfect)


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Fraud rate in test set: 2.5%
 
Always predict 'genuine'
  TN=5850  FP=0  FN=150  TP=0
  accuracy 0.975  precision 0.000  recall 0.000  F1 0.000
 
Random forest, threshold 0.5
  TN=5849  FP=1  FN=131  TP=19
  accuracy 0.978  precision 0.950  recall 0.127  F1 0.224
 
Threshold  precision  recall   flagged
   0.1      0.471    0.587     187
   0.2      0.790    0.527     100
   0.3      0.857    0.360      63
   0.5      0.950    0.127      20
   0.7      1.000    0.027       4
 
ROC AUC: 0.860   (0.5 = random guessing, 1.0 = perfect)
```

**What the output shows**

At the default threshold the random forest has 95% precision but catches only 13% of fraud. Lowering the threshold to 0.2 catches four times as much fraud at the cost of more false alarms. The right threshold is a business decision based on the relative cost of each type of error, not a purely technical one.

**Investigate**

1. Suppose each missed fraud costs £500 and each false alarm costs £10 in investigation time. Calculate the total cost at each threshold and choose the best.
2. Plot the precision-recall curve for this model using sklearn.metrics.PrecisionRecallDisplay.
3. Remove class_weight="balanced" and compare recall at the 0.5 threshold. What does the option do?

## 10. An end-to-end churn model

**File:** `l2_pipeline.py`

In [11]:
"""
Layer 2 example: a realistic end-to-end pipeline on messy tabular data.
File: l2_pipeline.py
 
Purpose:
  Real data has missing values, text categories and numbers on different
  scales. A ColumnTransformer applies the right preprocessing to each column,
  and a Pipeline guarantees identical processing at training and prediction
  time. We generate a synthetic customer churn dataset so no download is needed.
  This is the shape of most practical ML work: clean, encode, train, save.
 
How it works:
  1. Simulate 3,000 customers whose churn depends on tenure, support calls
     and contract type, then knock out some values to make the data messy.
  2. Build a ColumnTransformer: impute and scale the numbers; impute and
     one-hot encode the text categories.
  3. Chain it with a gradient boosting classifier in one Pipeline and train.
  4. Print a classification report on the held-out test set.
  5. Save the whole pipeline to disk, reload it and score a new customer.
 
What to look for in the output:
  - 255 missing bills and 180 missing contracts, yet training still works.
  - Churners are the minority class and harder to catch: recall 0.50 for
    'churned' against 0.82 for 'stayed'.
  - 5 raw columns become 12 model features after one-hot encoding,
    including a 'contract_missing' column.
  - The reloaded model gives 0.93 for a short-tenure, monthly customer with
    5 support calls, even though their bill is missing.
 
Requires: pip install numpy pandas scikit-learn joblib
"""
 
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report
import joblib
 
# ---- Simulate a customer table ---------------------------------------------
# Each column uses a distribution suited to its meaning: whole months, bills
# around 45 pounds, a Poisson count of support calls, and text categories.
rng = np.random.default_rng(5)
n = 3000
df = pd.DataFrame({
    "tenure_months": rng.integers(1, 72, n),
    "monthly_bill":  rng.normal(45, 15, n).round(2),
    "support_calls": rng.poisson(1.5, n),
    # p= gives the share of each contract type
    "contract":      rng.choice(["monthly", "annual", "two_year"], n,
                                p=[0.55, 0.3, 0.15]),
    "region":        rng.choice(["North", "Midlands", "South", "Wales",
                                 "Scotland"], n),
})
# The hidden 'true' behaviour: short tenure, many calls, monthly contracts churn
# The model never sees this formula; it must rediscover it from examples.
# Note that bill and region play no part, so they should matter little.
logit = (-0.04 * df.tenure_months + 0.5 * df.support_calls
         + 1.2 * (df.contract == "monthly") - 1.0)
# Pass the score through a sigmoid to get a churn probability, then draw
# a random outcome with that probability (real behaviour is noisy).
df["churned"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
 
# Make it messy: knock out 8% of bills and 5% of contract values
df.loc[rng.random(n) < 0.08, "monthly_bill"] = np.nan
df.loc[rng.random(n) < 0.05, "contract"] = np.nan
 
# Always inspect the data first: its shape, types and missing values.
print(df.head(), "\n")
print("Missing values per column:\n", df.isna().sum(), "\n")
 
# ---- Preprocessing: the right treatment for each column type ---------------
numeric = ["tenure_months", "monthly_bill", "support_calls"]
categorical = ["contract", "region"]
 
# Each entry is (name, transformer, columns it applies to).
preprocess = ColumnTransformer([
    # numbers: fill gaps with the median, then standardise
    # (the median is robust: a few extreme bills barely move it)
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    # categories: fill gaps with 'missing', then one-hot encode
    # One-hot encoding makes one 0/1 column per category, because models
    # need numbers and 'Wales' is not bigger or smaller than 'North'.
    # Keeping 'missing' as its own category lets the model learn whether
    # missingness itself is informative.
    # handle_unknown="ignore": a category never seen in training (e.g. a new
    # region) becomes all zeros instead of crashing at prediction time.
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant",
                                               fill_value="missing")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
     categorical),
])
 
# ---- One pipeline from raw table to prediction ------------------------------
# Because preprocessing lives INSIDE the pipeline, the medians, means and
# category lists are learned from the training rows only (no data leakage),
# and exactly the same steps are replayed on any new data.
# HistGradientBoostingClassifier: fast gradient boosted trees that bin
# numeric values into histograms, well suited to tabular data.
pipe = Pipeline([("prep", preprocess),
                 ("model", HistGradientBoostingClassifier(random_state=0))])
 
# X keeps every column except the target; y is the churn label.
X, y = df.drop(columns="churned"), df["churned"]
# stratify=y keeps the churn rate equal in the training and test sets
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y,
                                          random_state=0)
pipe.fit(X_tr, y_tr)
# classification_report gives precision, recall and F1 for each class;
# 'support' is the number of test customers in that class.
print(classification_report(y_te, pipe.predict(X_te),
                            target_names=["stayed", "churned"]))
 
# One-hot encoding turned 2 text columns into many binary columns:
# (names are prefixed by the ColumnTransformer step: num__ or cat__)
print("Features after preprocessing:",
      list(pipe.named_steps["prep"].get_feature_names_out()))
 
# ---- Deploy: save the whole pipeline, reload it, predict a new customer -----
# Saving the pipeline (not just the model) means the deployed system cannot
# forget a preprocessing step or apply it differently from training.
joblib.dump(pipe, "churn_model.joblib")
loaded = joblib.load("churn_model.joblib")
# The new customer's bill is unknown (NaN): the pipeline imputes it with the
# training median automatically.
new_customer = pd.DataFrame([{"tenure_months": 3, "monthly_bill": np.nan,
                              "support_calls": 5,
                              "contract": "monthly", "region": "Wales"}])
# [0, 1] -> first (only) row, column 1 = probability of churn
print(f"\nChurn probability for new customer: "
      f"{loaded.predict_proba(new_customer)[0, 1]:.2f}")

   tenure_months  monthly_bill  support_calls  contract    region  churned
0             48         69.64              0  two_year  Scotland        0
1             58         30.73              2   monthly     Wales        0
2              2         72.36              2   monthly     Wales        1
3             58           NaN              1  two_year     Wales        0
4             34          7.71              2   monthly     Wales        0 

Missing values per column:
 tenure_months      0
monthly_bill     255
support_calls      0
contract         180
region             0
churned            0
dtype: int64 



              precision    recall  f1-score   support

      stayed       0.78      0.82      0.80       510
     churned       0.56      0.50      0.53       240

    accuracy                           0.71       750
   macro avg       0.67      0.66      0.66       750
weighted avg       0.71      0.71      0.71       750

Features after preprocessing: ['num__tenure_months', 'num__monthly_bill', 'num__support_calls', 'cat__contract_annual', 'cat__contract_missing', 'cat__contract_monthly', 'cat__contract_two_year', 'cat__region_Midlands', 'cat__region_North', 'cat__region_Scotland', 'cat__region_South', 'cat__region_Wales']

Churn probability for new customer: 0.93


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
   tenure_months  monthly_bill  support_calls  contract    region  churned
0             48         69.64              0  two_year  Scotland        0
1             58         30.73              2   monthly     Wales        0
2              2         72.36              2   monthly     Wales        1
3             58           NaN              1  two_year     Wales        0
4             34          7.71              2   monthly     Wales        0 
 
Missing values per column:
 tenure_months      0
monthly_bill     255
support_calls      0
contract         180
region             0
churned            0
dtype: int64 
 
              precision    recall  f1-score   support
 
      stayed       0.78      0.82      0.80       510
     churned       0.56      0.50      0.53       240
 
    accuracy                           0.71       750
   macro avg       0.67      0.66      0.66       750
weighted avg       0.71      0.71      0.71       750
 
Features after preprocessing: ['num__tenure_months', 'num__monthly_bill', 'num__support_calls', 'cat__contract_annual', 'cat__contract_missing', 'cat__contract_monthly', 'cat__contract_two_year', 'cat__region_Midlands', 'cat__region_North', 'cat__region_Scotland', 'cat__region_South', 'cat__region_Wales']
 
Churn probability for new customer: 0.93
```

**Investigate**

1. Predict for a customer with region "Northern Ireland", a value never seen during training. Why does the pipeline not crash? Which parameter is responsible?
2. Add a feature "calls_per_year" = support_calls / tenure_months \* 12 using sklearn.preprocessing.FunctionTransformer inside the pipeline.
3. Deliberately introduce leakage: add a column "cancellation_letter_sent" copied from churned with 5% noise. Observe the suspiciously high score and explain why it would fail in production.

## 11. Q-learning in a grid world

**File:** `l2_qlearning.py`

In [12]:
"""
Layer 2 example: reinforcement learning with tabular Q-learning.
File: l2_qlearning.py
 
Purpose:
  Reinforcement learning is the third kind of machine learning, alongside
  supervised and unsupervised: an agent learns by trial and error from
  rewards, with no labelled examples. Q-learning is the classic algorithm
  and the ancestor of the deep RL used in game-playing systems.
 
  An agent in a 4 x 4 grid world must reach the goal (G) and avoid the pit (P).
  It is never told the rules; it learns only from REWARDS:
    +10 for the goal, -10 for the pit, -1 for every other step (hurry up!)
 
  Q(s, a) estimates the long-term reward of taking action a in state s.
  After each step the Q-learning update is:
    Q(s,a) <- Q(s,a) + alpha * (r + gamma * max_a' Q(s',a') - Q(s,a))
 
How it works:
  1. Define the grid, the four moves and the reward for each move.
  2. Start with a table of Q-values that are all zero.
  3. Play 2,000 episodes from the top-left corner, choosing moves by
     epsilon-greedy and updating the table after every step.
  4. Print the best action in each square: the learned policy.
 
What to look for in the output:
  - Early episodes wander (reward -17 or -18); by episode 2000 the agent
    reaches the goal in the minimum 6 steps (reward 5 = 10 - 5 x 1).
  - Epsilon starts at 1.0 (printed as 0.99 after episode 1) and falls to
    0.05: exploration gives way to exploitation.
  - The arrows lead to G from every square, and those next to the pit
    steer around it.
 
Requires: pip install numpy
"""
 
import numpy as np
 
# ---- The environment -------------------------------------------------------
# A state is a (row, column) pair; (0, 0) is the top-left corner.
rng = np.random.default_rng(0)
SIZE = 4
GOAL, PIT = (3, 3), (1, 2)
# each action is a (row change, column change)
ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # up, down, left, right
ARROWS = "^v<>"
 
 
# step() is the environment's rules: given a state and action it returns the
# next state, the reward and whether the episode has ended. The agent can
# only call it and observe the result; it never reads this code.
def step(state, action):
    r, c = state
    dr, dc = ACTIONS[action]
    # min/max clamp the move to the grid, so walking into a wall stays put
    nxt = (min(max(r + dr, 0), SIZE - 1), min(max(c + dc, 0), SIZE - 1))
    if nxt == GOAL:
        return nxt, 10, True
    if nxt == PIT:
        return nxt, -10, True
    return nxt, -1, False
 
 
# ---- The agent's knowledge: a Q-table ----------------------------------------
# alpha: how far each update moves Q towards the new target (0.1 = 10%).
# gamma: how much future reward counts compared with reward now; 0.95 makes
# the agent far-sighted enough to accept -1 steps on the way to +10.
Q = np.zeros((SIZE, SIZE, 4))          # one value per state and action
alpha, gamma = 0.1, 0.95                # learning rate, discount factor
epsilon = 1.0                           # exploration rate, decays over time
 
# ---- Training loop -----------------------------------------------------------
# One episode = one attempt from the start square until the goal, the pit,
# or a 50-step limit (so a lost agent cannot wander forever).
for episode in range(1, 2001):
    state, total, done, steps = (0, 0), 0, False, 0
    while not done and steps < 50:
        # EPSILON-GREEDY: explore randomly sometimes (probability epsilon),
        # otherwise exploit the best known action.
        # Without exploration the agent would repeat its first lucky path and
        # never discover better ones; without exploitation it never improves.
        if rng.random() < epsilon:
            action = int(rng.integers(4))
        else:
            action = int(Q[state].argmax())
        nxt, reward, done = step(state, action)
        # target: reward now + discounted best future value (0 if episode ended)
        target = reward + (0 if done else gamma * Q[nxt].max())
        # The Q-learning update: move Q(s, a) a fraction alpha of the way
        # towards the target. (target - Q) is the 'temporal difference' error:
        # how much better or worse things went than expected.
        Q[state][action] += alpha * (target - Q[state][action])
        state, total, steps = nxt, total + reward, steps + 1
    # Decay exploration after each episode, but never below 5%, so the agent
    # keeps occasionally testing alternatives.
    epsilon = max(0.05, epsilon * 0.995)
    if episode in (1, 50, 200, 2000):
        print(f"episode {episode:4d}  reward {total:4d}  steps {steps:2d}  "
              f"epsilon {epsilon:.2f}")
 
# ---- Read off the policy -----------------------------------------------------
# The POLICY is simply: in each square, take the action with the highest Q.
print("\nLearned policy (best action in each square):")
for r in range(SIZE):
    row = ""
    for c in range(SIZE):
        row += (" G " if (r, c) == GOAL else " P " if (r, c) == PIT
                else f" {ARROWS[Q[r, c].argmax()]} ")
    print(row)

episode    1  reward  -17  steps  8  epsilon 0.99
episode   50  reward  -18  steps  9  epsilon 0.78


episode  200  reward    0  steps 11  epsilon 0.37


episode 2000  reward    5  steps  6  epsilon 0.05



Learned policy (best action in each square):
 v  v  >  v 
 v  v  P  v 
 v  v  v  v 
 >  >  >  G 


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
episode    1  reward  -17  steps  8  epsilon 0.99
episode   50  reward  -18  steps  9  epsilon 0.78
episode  200  reward    0  steps 11  epsilon 0.37
episode 2000  reward    5  steps  6  epsilon 0.05
 
Learned policy (best action in each square):
 v  v  >  v 
 v  v  P  v 
 v  v  v  v 
 >  >  >  G
```

**What the output shows**

The agent was never told where the goal or pit was, or that walls exist. Early episodes wander and score negative rewards; by the end it reaches the goal in the minimum six steps. The learned policy also steers around the pit: from the square directly above the pit, the agent moves right rather than down.

**Investigate**

1. Set gamma to 0.1. The agent becomes short-sighted. Does it still learn a sensible route? Why?
2. Change the step reward from -1 to 0. How does the learned path length change, and why?
3. Make the environment "slippery": with 20% probability the agent moves in a random direction. Does the policy become more cautious near the pit?

## 12. Bias through a proxy

**File:** `l2_fairness.py`

In [13]:
"""
Layer 2 example: how a model learns bias through a PROXY feature.
File: l2_fairness.py
 
Purpose:
  Removing a protected attribute (such as ethnicity or sex) from the data
  does not make a model fair. Other features that correlate with it, called
  PROXIES, let the model reconstruct it, and biased historic labels teach the
  model to repeat past discrimination. This matters for any model used to
  make decisions about people.
 
  We simulate historic hiring decisions that were unfair to Group B.
  The protected attribute 'group' is REMOVED before training, yet the
  model still discriminates, because 'postcode_area' correlates with group.
  All data is synthetic and for teaching purposes only.
 
How it works:
  1. Simulate 10,000 applicants: skill and experience are equal across
     groups, but past hiring decisions penalised Group B.
  2. Train logistic regression WITHOUT the group column.
  3. Compare selection rates and true positive rates between the groups.
  4. Compute the disparate impact ratio, then retrain without the proxy.
 
What to look for in the output:
  - Historic hire rate: 0.594 for Group A against 0.241 for Group B.
  - The postcode weight is strongly negative (-1.899), although postcode
    has nothing to do with ability: the model has learned the proxy.
  - Disparate impact ratio 0.69, below the 0.8 warning level.
  - Removing postcode lifts the ratio to 1.02, but the labels are still biased.
 
Requires: pip install numpy pandas scikit-learn
"""
 
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
 
# ---- Simulate applicants ----------------------------------------------------
rng = np.random.default_rng(11)
n = 10000
group = rng.choice(["A", "B"], n)
# skill is equally distributed in both groups
skill = rng.normal(50, 10, n)
# postcode_area is 1 for 80% of group B and 20% of group A: a proxy
# (in real life, where people live often correlates with ethnicity or income)
postcode_area = np.where(group == "B", rng.random(n) < 0.8,
                         rng.random(n) < 0.2).astype(int)
experience = rng.normal(5, 2, n)
 
# Historic decisions: based on skill, but Group B was penalised by past recruiters
# The -12 is the unfair penalty. These biased decisions become the training
# LABELS, so a model that fits them well also learns the penalty.
hired = (skill + 3 * experience - 12 * (group == "B")
         + rng.normal(0, 5, n) > 62).astype(int)
 
df = pd.DataFrame({"skill": skill, "experience": experience,
                   "postcode_area": postcode_area, "group": group, "hired": hired})
print("Historic hire rate by group:\n",
      df.groupby("group")["hired"].mean().round(3), "\n")
 
# ---- Train a model that never sees 'group' ----------------------------------
# 'Fairness through unawareness': drop the group column and hope for the best
features = ["skill", "experience", "postcode_area"]
tr, te = train_test_split(df, test_size=0.3, random_state=0)
model = LogisticRegression(max_iter=1000).fit(tr[features], tr["hired"])
# add a 'pred' column holding the model's 0/1 decision for each test applicant
te = te.assign(pred=model.predict(te[features]))
 
# The model is rewarded for matching the biased labels. Since postcode is
# the only available clue to group, it uses postcode to recreate the penalty.
print("Learned weights:", dict(zip(features, model.coef_[0].round(3))))
print("The negative postcode weight shows the model has learned the proxy.\n")
 
# ---- Measure fairness by group ----------------------------------------------
# Selection rate: share of each group the model would hire.
# There are several fairness definitions and they can conflict; here Group B
# has a lower selection rate but a higher TPR, so which looks 'fair' depends
# on the measure chosen.
print("Selection rate and true positive rate by group (test set):")
for g, sub in te.groupby("group"):
    sel = sub["pred"].mean()
    # TPR: of the people historically hired, how many does the model select?
    tpr = sub.loc[sub.hired == 1, "pred"].mean()
    print(f"  Group {g}: selection rate {sel:.3f}   true positive rate {tpr:.3f}")
 
rates = te.groupby("group")["pred"].mean()
# Disparate impact ratio: below 0.8 is a common warning sign (the 'four-fifths rule')
print(f"\nDisparate impact ratio (B / A): {rates['B'] / rates['A']:.2f}")
 
# ---- What happens if we also remove the proxy? ------------------------------
# With only skill and experience, which are equal across groups, the model
# has no route back to group membership.
model2 = LogisticRegression(max_iter=1000).fit(tr[["skill", "experience"]],
                                               tr["hired"])
te2 = te.assign(pred=model2.predict(te[["skill", "experience"]]))
r2 = te2.groupby("group")["pred"].mean()
print(f"Without postcode: disparate impact ratio {r2['B'] / r2['A']:.2f}")
# In real data there are usually many weak proxies, so removing them one by
# one is rarely enough; the deeper fix is to question the labels themselves.
print("Bias is reduced, but the labels themselves still encode past unfairness.")

Historic hire rate by group:
 group
A    0.594
B    0.241
Name: hired, dtype: float64 



Learned weights: {'skill': np.float64(0.247), 'experience': np.float64(0.733), 'postcode_area': np.float64(-1.899)}
The negative postcode weight shows the model has learned the proxy.

Selection rate and true positive rate by group (test set):
  Group A: selection rate 0.486   true positive rate 0.746
  Group B: selection rate 0.333   true positive rate 0.885

Disparate impact ratio (B / A): 0.69


Without postcode: disparate impact ratio 1.02
Bias is reduced, but the labels themselves still encode past unfairness.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Historic hire rate by group:
 group
A    0.594
B    0.241
Name: hired, dtype: float64 
 
Learned weights: {'skill': np.float64(0.247), 'experience': np.float64(0.733), 'postcode_area': np.float64(-1.899)}
The negative postcode weight shows the model has learned the proxy.
 
Selection rate and true positive rate by group (test set):
  Group A: selection rate 0.486   true positive rate 0.746
  Group B: selection rate 0.333   true positive rate 0.885
 
Disparate impact ratio (B / A): 0.69
Without postcode: disparate impact ratio 1.02
Bias is reduced, but the labels themselves still encode past unfairness.
```

**What the output shows**

Even though the group column was removed, the model learned a strong negative weight for postcode area and selected Group B at only 69% of the rate of Group A, below the four-fifths guideline. Removing the proxy equalised selection rates in this synthetic case, but real datasets usually contain many weaker proxies, and the historic labels themselves still reflect past unfairness.

**Investigate**

1. Calculate the false positive rate for each group. Are the two groups treated equally on this measure?
2. Research the fairlearn library and use its MetricFrame to produce the same per-group analysis.
3. Write a short model card for the hiring model describing its intended use, training data, limitations and group performance.